In [30]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Input

In [32]:
# Verileri tekrar oku
df = pd.read_csv("all_data_ratio.csv")

# Hedef değişkenler ve feature eşlemesi
target_variables = ["uzman", "pratisyen", "asistan", "ebe", "diger", "hemsire", "yatak", "hastane"]

feature_map = {
    "uzman": ["yıl", "nufus", "hastane", "yatak", "asistan", "pratisyen", "hemsire", "ebe", "diger"],
    "pratisyen": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "hemsire", "ebe", "diger"],
    "asistan": ["yıl", "nufus", "hastane", "yatak", "uzman", "pratisyen", "hemsire", "ebe", "diger"],
    "ebe": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "hemsire", "pratisyen", "diger"],
    "diger": ["yıl", "nufus", "hastane", "yatak", "asistan", "pratisyen", "hemsire", "ebe", "uzman"],
    "hemsire": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "pratisyen", "ebe", "diger"],
    "yatak": ["yıl", "nufus", "hastane", "hemsire", "asistan", "uzman", "pratisyen", "ebe", "diger"],
    "hastane": ["yıl", "nufus", "yatak", "hemsire", "asistan", "uzman", "pratisyen", "ebe", "diger"]
}

results = []

In [34]:
# GRU tahmin fonksiyonu
def forecast_gru(X, y):
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()
    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y.reshape(-1, 1))

    X_gru = np.reshape(X_scaled, (X_scaled.shape[0], 1, X_scaled.shape[1]))

    model = Sequential([
        Input(shape=(X_gru.shape[1], X_gru.shape[2])),
        GRU(50),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    model.fit(X_gru[:-1], y_scaled[:-1], epochs=20, verbose=0)

    X_pred = np.reshape(X_scaled[-1:], (1, 1, X_scaled.shape[1]))
    prediction_scaled = model.predict(X_pred, verbose=0)[0][0]
    prediction = scaler_y.inverse_transform([[prediction_scaled]])[0][0]
    return prediction

In [36]:
# Tahmin döngüsü
for il in df['il_adı'].unique():
    il_data = df[df['il_adı'] == il]
    for target in target_variables:
        features = feature_map[target]
        columns_to_use = list(set(features + [target]))
        il_filtered = il_data[columns_to_use]
        il_filtered = il_filtered[il_filtered['yıl'] <= 2022]

        if len(il_filtered) < 5:
            continue

        y_series = il_filtered[target]

        try:
            X = il_filtered.drop(columns=[target]).values
            y = y_series.values
            if np.isnan(X).any() or np.isnan(y).any():
                continue
            gru_pred = forecast_gru(X, y)
        except Exception as e:
            gru_pred = None

        results.append({
            'il': il,
            'target': target,
            'gru': gru_pred
        })

KeyboardInterrupt: 

In [9]:
# Sonuçları DataFrame olarak kaydet
results_df = pd.DataFrame(results)
results_df.to_csv("tahmin_sonuclari_2023.csv", index=False)

# Pivot yaparak şehir bazlı görünüm oluştur
pivot_df = results_df.pivot(index='il', columns='target', values='gru')
pivot_df.reset_index(inplace=True)
pivot_df.to_csv("sehir_bazli_tahminler_2023.csv", index=False)

import ace_tools as tools; tools.display_dataframe_to_user(name="Şehir Bazlı GRU Tahminleri", dataframe=pivot_df)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Tahminleri oku
pred_df = pd.read_csv("tahmin_sonuclari_2023.csv")

# Gerçek veriyi oku
real_df = pd.read_csv("all_data_ratio.csv")
real_df = real_df[real_df['yıl'] == 2023]

# Sonuçları saklamak için liste
metrics_list = []

# Kullanılacak modeller
models = ["arima", "prophet", "exp_smoothing", "gru", "seq2seq"]
target_vars = pred_df['target'].unique()

for target in target_vars:
    for model in models:
        # Model tahminlerini ve gerçek veriyi eşleştir
        pred_subset = pred_df[pred_df['target'] == target][['il', model]].dropna()
        real_subset = real_df[['il_adı', target]].rename(columns={'il_adı': 'il'})
        merged = pd.merge(pred_subset, real_subset, on='il')
        
        if len(merged) < 5:  # Sağlıklı istatistik için minimum örnek
            continue
        
        y_true = merged[target].values
        y_pred = merged[model].values

        # Metrikleri hesapla
        r2 = r2_score(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)
        rmse = mean_squared_error(y_true, y_pred, squared=False)
        mape = mean_absolute_percentage_error(y_true, y_pred)

        metrics_list.append({
            'target': target,
            'model': model,
            'R2': round(r2, 4),
            'MAE': round(mae, 2),
            'RMSE': round(rmse, 2),
            'MAPE (%)': round(mape * 100, 2)
        })

# Sonuçları DataFrame'e çevir ve kaydet
metrics_df = pd.DataFrame(metrics_list)
metrics_df.to_csv("model_dogruluk_skorlari.csv", index=False)

# Ekrana yazdır
import ace_tools as tools; tools.display_dataframe_to_user(name="Model Doğruluk Skorları", dataframe=metrics_df)

In [ ]:
import pandas as pd
from sklearn.metrics import mean_absolute_percentage_error

# Bölge haritası
bolge_haritasi = {
    'Marmara': ["İstanbul", "Edirne", "Kırklareli", "Tekirdağ", "Kocaeli", "Sakarya", "Yalova", "Bilecik", "Balıkesir", "Çanakkale", "Bursa"],
    'Ege': ["İzmir", "Aydın", "Muğla", "Manisa", "Afyonkarahisar", "Uşak", "Kütahya", "Denizli"],
    'Akdeniz': ["Antalya", "Mersin", "Adana", "Hatay", "Kahramanmaraş", "Osmaniye", "Isparta", "Burdur"],
    'İç Anadolu': ["Ankara", "Konya", "Kayseri", "Eskişehir", "Nevşehir", "Aksaray", "Kırıkkale", "Niğde", "Karaman", "Sivas", "Yozgat", "Kırşehir", "Çankırı"],
    'Karadeniz': ["Samsun", "Trabzon", "Ordu", "Giresun", "Rize", "Artvin", "Tokat", "Amasya", "Bartın", "Zonguldak", "Karabük", "Sinop", "Çorum", "Bayburt", "Gümüşhane", "Kastamonu", "Bolu", "Düzce"],
    'Doğu Anadolu': ["Erzurum", "Van", "Malatya", "Elazığ", "Kars", "Ağrı", "Bitlis", "Iğdır", "Tunceli", "Ardahan", "Erzincan", "Bingöl", "Muş", "Hakkari"],
    'Güneydoğu Anadolu': ["Diyarbakır", "Gaziantep", "Şanlıurfa", "Mardin", "Batman", "Siirt", "Şırnak", "Adıyaman", "Kilis"]
}

# Bölge eşlemesi
bolge_df = pd.DataFrame([
    {'il': sehir, 'bolge': bolge}
    for bolge, iller in bolge_haritasi.items()
    for sehir in iller
])

# Verileri oku
tahminler = pd.read_csv("tahmin_sonuclari_2023.csv")
gercekler = pd.read_csv("all_data_ratio.csv")

# Gerçek verileri 2023 yılı için al ve uzun forma çevir
gercekler = gercekler[gercekler["yıl"] == 2023]
gercekler.rename(columns={"il_adı": "il"}, inplace=True)
gercekler = gercekler.melt(id_vars=["il"], var_name="target", value_name="gercek")

# Birleştir
birlesik = pd.merge(tahminler, gercekler, on=["il", "target"], how="inner")
birlesik = pd.merge(birlesik, bolge_df, on="il", how="left")

# Bölge bazlı prophet/gru/seq2seq ve gerçek toplamları
bolgesel_ortalamalar = birlesik.groupby(["bolge", "target"]).agg({
    "gercek": "sum",
    "prophet": "sum",
    "gru": "sum",
    "seq2seq": "sum"
}).reset_index()

# Doğruluk hesapla
bolgesel_ortalamalar["prophet_accuracy_%"] = 100 - mean_absolute_percentage_error(
    bolgesel_ortalamalar["gercek"], bolgesel_ortalamalar["prophet"]) * 100
bolgesel_ortalamalar["gru_accuracy_%"] = 100 - mean_absolute_percentage_error(
    bolgesel_ortalamalar["gercek"], bolgesel_ortalamalar["gru"]) * 100
bolgesel_ortalamalar["seq2seq_accuracy_%"] = 100 - mean_absolute_percentage_error(
    bolgesel_ortalamalar["gercek"], bolgesel_ortalamalar["seq2seq"]) * 100

# Toplam doğruluk
toplam_prophet_accuracy = 100 - mean_absolute_percentage_error(birlesik["gercek"], birlesik["prophet"]) * 100
toplam_gru_accuracy = 100 - mean_absolute_percentage_error(birlesik["gercek"], birlesik["gru"]) * 100
toplam_seq2seq_accuracy = 100 - mean_absolute_percentage_error(birlesik["gercek"], birlesik["seq2seq"]) * 100

# CSV olarak kaydet
csv_path = "bolgesel_karsilastirma.csv"
bolgesel_ortalamalar.to_csv(csv_path, index=False)

# Göster
import ace_tools as tools
tools.display_dataframe_to_user(name="Bölgesel Prophet, GRU ve Seq2Seq Karşılaştırması", dataframe=bolgesel_ortalamalar)

# Genel çıktılar
print("Toplam Prophet doğruluğu: %", round(toplam_prophet_accuracy, 2))
print("Toplam GRU doğruluğu: %", round(toplam_gru_accuracy, 2))
print("Toplam Seq2Seq doğruluğu: %", round(toplam_seq2seq_accuracy, 2))